# 05 — Model Research & Hypothesis Testing

Sandbox notebook. No models saved here. Pure research.

## Tests
1. **Retrain 2009-2018 / Test 2019-2021** — fresh split, never-seen regime
2. **Proper momentum baseline** — linear regression, stronger than naive
3. **Feature importance** — what is actually driving predictions?
4. **Prediction stability** — does confidence vary or is it always 99%?
5. **Pair-by-pair breakdown** — where is the edge concentrated?
6. **Regime sensitivity** — does the model actually update when market changes?

In [ ]:
# ── Imports & Setup ────────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
from pathlib import Path
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

DARK = '#080c14'
BLUE = '#4fc3f7'
GOLD = '#ffd700'
RED  = '#ef5350'
GREEN = '#66bb6a'

# ── Paths ──────────────────────────────────────────────────────
DATA_DIR   = Path('../backend/data/features')
MODELS_DIR = Path('../backend/models')

HORIZONS       = ['1H', '4H', '1D', '7D']
QUANTILES      = [0.10, 0.25, 0.50, 0.75, 0.90]
QUANTILE_NAMES = ['Q10', 'Q25', 'Q50', 'Q75', 'Q90']
PAIRS          = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF', 'AUDUSD', 'USDCAD', 'NZDUSD']

# ── Load data ──────────────────────────────────────────────────
print('Loading features...')
df = pd.read_parquet(DATA_DIR / 'all_pairs_features_labels.parquet')
df.index = pd.to_datetime(df.index)
print(f'Loaded: {len(df):,} rows | {df.index.min().date()} → {df.index.max().date()}')

feature_cols = [c for c in df.columns if not c.startswith('label_') and c != 'pair']
print(f'Features: {len(feature_cols)}')

In [ ]:
# ── Shared helpers ─────────────────────────────────────────────

def derive_p_down(q_vals):
    qs = np.array(QUANTILES)
    vals = np.sort(q_vals)
    if vals[0] <= 0 <= vals[-1]:
        return float(np.interp(0, vals, qs))
    elif vals[-1] < 0:
        slope = (qs[-1] - qs[-2]) / (vals[-1] - vals[-2] + 1e-10)
        return float(np.clip(qs[-1] + slope * (0 - vals[-1]), 0.90, 0.999))
    else:
        slope = (qs[1] - qs[0]) / (vals[1] - vals[0] + 1e-10)
        return float(np.clip(qs[0] + slope * (0 - vals[0]), 0.001, 0.10))


def get_predictions(df_subset, horizon, models, calibrator):
    """Run inference on a DataFrame subset. Returns p_cal, p_dom, correct arrays."""
    X = df_subset[feature_cols].ffill()
    y = df_subset[f'label_{horizon}'].values

    q_preds = {}
    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        q_preds[q_name] = models[(horizon, q_name)].predict(X)

    p_raw = np.array([
        derive_p_down(np.array([q_preds[n][i] for n in QUANTILE_NAMES]))
        for i in range(len(X))
    ])

    p_cal = calibrator.predict(p_raw)
    p_dom = np.where(p_cal > 0.5, p_cal, 1 - p_cal)
    correct = np.where(p_cal > 0.5, y < 0, y > 0)

    return p_cal, p_dom, correct, y


def accuracy_at_threshold(p_dom, correct, threshold=0.70):
    mask = p_dom >= threshold
    if mask.sum() < 10:
        return float('nan'), 0
    return correct[mask].mean(), mask.sum()


print('Helpers ready.')

---
## TEST 1 — Retrain on 2009-2018, Evaluate on 2019-2021

**Hypothesis:** If the model has genuine structural edge, it will generalize to a completely different macro regime (pre-COVID, different Fed cycle, trade war) that it was NOT trained on in the original setup.

**Setup:**
- Train: 2009-2018 (walk-forward, same as notebook 03)
- Calibration: 2018-2019
- Test: 2019-2021 (includes COVID crash — ultimate stress test)

**We do NOT save these models.** Research only.

In [ ]:
# TEST 1 — Fresh split retrain
# Runtime: ~2-3 hours on GPU. Start this and come back.

import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression

TRAIN_END  = '2018-12-31'
CAL_END    = '2019-12-31'
TEST_END   = '2021-12-31'

df_train = df[df.index <= TRAIN_END]
df_cal   = df[(df.index > TRAIN_END) & (df.index <= CAL_END)]
df_test1 = df[(df.index > CAL_END)  & (df.index <= TEST_END)]

print(f'Train:       {len(df_train):,} rows  ({df_train.index.min().date()} → {df_train.index.max().date()})')
print(f'Calibration: {len(df_cal):,} rows  ({df_cal.index.min().date()} → {df_cal.index.max().date()})')
print(f'Test:        {len(df_test1):,} rows  ({df_test1.index.min().date()} → {df_test1.index.max().date()})')
print(f'\nTest period includes: 2019 (pre-COVID), 2020 (COVID crash), 2021 (recovery)')
print('This is the hardest possible out-of-sample test.')

In [ ]:
def get_lgbm_params_t1(quantile):
    return {
        'objective':        'quantile',
        'alpha':            quantile,
        'n_estimators':     3000,
        'learning_rate':    0.02,
        'num_leaves':       127,
        'min_child_samples': 50,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'device':           'gpu',
        'verbosity':        -1,
    }

X_train = df_train[feature_cols].ffill()
X_cal   = df_cal[feature_cols].ffill()
X_test1 = df_test1[feature_cols].ffill()

t1_models      = {}   # {(horizon, q_name): model}
t1_calibrators = {}   # {horizon: IsotonicRegression}

for horizon in HORIZONS:
    y_train = df_train[f'label_{horizon}'].values
    y_cal   = df_cal[f'label_{horizon}'].values
    print(f'\nTraining {horizon}...')

    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        model = lgb.LGBMRegressor(**get_lgbm_params_t1(q))
        model.fit(
            X_train, y_train,
            eval_set=[(X_cal, y_cal)],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(500),
            ]
        )
        t1_models[(horizon, q_name)] = model
        print(f'  {q_name}: {model.best_iteration_} iters')

    # Calibrate on 2018-2019
    q_preds_cal = {}
    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        q_preds_cal[q_name] = t1_models[(horizon, q_name)].predict(X_cal)

    p_raw_cal = np.array([
        derive_p_down(np.array([q_preds_cal[n][i] for n in QUANTILE_NAMES]))
        for i in range(len(X_cal))
    ])

    y_binary_cal = (y_cal < 0).astype(float)
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(p_raw_cal, y_binary_cal)
    t1_calibrators[horizon] = iso
    print(f'  Calibrator fitted on {len(y_cal):,} cal samples.')

print('\nAll models trained and calibrated.')

In [ ]:
# TEST 1 — Evaluate on 2019-2021 (never seen by model)

print('TEST 1 RESULTS — Train: 2009-2018 | Test: 2019-2021')
print('=' * 65)
print(f'Includes COVID crash (Feb-Mar 2020) — ultimate stress test')
print()

# Also compare with original model on same period
orig_models = {}
orig_calibrators = {}
for horizon in HORIZONS:
    orig_calibrators[horizon] = joblib.load(MODELS_DIR / f'calibrator_{horizon}.joblib')
    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        bundle = joblib.load(MODELS_DIR / f'model_{horizon}_Q{int(q*100)}.joblib')
        orig_models[(horizon, q_name)] = bundle['model']

print(f'{"Horizon":<8} {"Threshold":<12} {"T1 acc":<12} {"T1 freq":<12} {"Orig acc":<12} {"Orig freq"}')
print('-' * 68)

df_eurusd_test1 = df_test1[df_test1['pair'] == 'EURUSD']

for horizon, step in [('1H',1),('4H',4),('1D',24),('7D',168)]:
    df_sub = df_eurusd_test1.iloc[::step]

    # New model
    p_cal_t1, p_dom_t1, correct_t1, _ = get_predictions(
        df_sub, horizon, t1_models, t1_calibrators[horizon]
    )

    # Original model on same period
    p_cal_or, p_dom_or, correct_or, _ = get_predictions(
        df_sub, horizon, orig_models, orig_calibrators[horizon]
    )

    for thresh in [0.65, 0.70, 0.75]:
        acc_t1, n_t1 = accuracy_at_threshold(p_dom_t1, correct_t1, thresh)
        acc_or, n_or = accuracy_at_threshold(p_dom_or, correct_or, thresh)
        freq_t1 = (p_dom_t1 >= thresh).mean()
        freq_or = (p_dom_or >= thresh).mean()
        print(f'{horizon:<8} >={thresh:.0%}      {acc_t1:<12.1%} {freq_t1:<12.1%} {acc_or:<12.1%} {freq_or:.1%}')
    print()

---
## TEST 2 — Feature Importance & Prediction Stability

**Hypothesis A:** If `hour` dominates feature importance, the model is mostly a session timer — real but limited.

**Hypothesis B:** If confidence is always 99%, the model is stuck in one regime and not actually discriminating.

In [ ]:
# TEST 2A — Feature importance from original models

print('Feature Importance Analysis — Original Models')
print('=' * 55)

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.patch.set_facecolor(DARK)

for idx, horizon in enumerate(HORIZONS):
    ax = axes[idx // 2][idx % 2]
    ax.set_facecolor(DARK)

    # Average importance across 5 quantile models
    importance_sum = np.zeros(len(feature_cols))
    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        bundle = joblib.load(MODELS_DIR / f'model_{horizon}_Q{int(q*100)}.joblib')
        importance_sum += bundle['model'].feature_importances_

    importance_avg = importance_sum / len(QUANTILES)
    top_idx = np.argsort(importance_avg)[-20:][::-1]
    top_features = [feature_cols[i] for i in top_idx]
    top_values   = importance_avg[top_idx]

    colors = [RED if 'hour' in f or 'session' in f or 'day_of' in f or 'is_mon' in f or 'is_fri' in f or 'month' in f
              else BLUE for f in top_features]

    ax.barh(range(20), top_values[::-1], color=colors[::-1])
    ax.set_yticks(range(20))
    ax.set_yticklabels(top_features[::-1], fontsize=7, color='white')
    ax.set_title(f'{horizon} — Top 20 Features', color='white', fontsize=10)
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#1a2332')

    # Print time feature share
    time_features = ['hour', 'session_asian', 'session_london', 'session_ny',
                     'session_overlap', 'day_of_week', 'is_monday', 'is_friday', 'month']
    time_mask = np.array([any(tf in fc for tf in time_features) for fc in feature_cols])
    time_share = importance_avg[time_mask].sum() / importance_avg.sum()
    print(f'{horizon}: time features = {time_share:.1%} of total importance')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=RED, label='Time/session features'),
                   Patch(facecolor=BLUE, label='Price/technical features')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           facecolor=DARK, labelcolor='white', fontsize=10)

plt.suptitle('Feature Importance by Horizon\nRed = time/session features', color='white', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# TEST 2B — Prediction stability over time
# Does confidence vary meaningfully or is it always 99%?

print('Prediction Stability Analysis — EURUSD, Test Set 2022-2025')
print('=' * 60)

df_test = df[df.index >= '2022-01-01']
df_eu   = df_test[df_test['pair'] == 'EURUSD']

fig, axes = plt.subplots(4, 1, figsize=(18, 16))
fig.patch.set_facecolor(DARK)

for idx, horizon in enumerate(HORIZONS):
    ax = axes[idx]
    ax.set_facecolor(DARK)

    bundle = joblib.load(MODELS_DIR / f'model_{horizon}_Q50.joblib')
    orig_models_single = {}
    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        b = joblib.load(MODELS_DIR / f'model_{horizon}_Q{int(q*100)}.joblib')
        orig_models_single[(horizon, q_name)] = b['model']

    cal = joblib.load(MODELS_DIR / f'calibrator_{horizon}.joblib')
    p_cal, p_dom, correct, y = get_predictions(df_eu, horizon, orig_models_single, cal)

    # Plot confidence over time
    ax.plot(df_eu.index, p_dom, color=BLUE, alpha=0.4, linewidth=0.5)
    ax.axhline(0.70, color=GOLD, linewidth=1, linestyle='--', alpha=0.7)
    ax.axhline(0.50, color='white', linewidth=0.5, linestyle='--', alpha=0.3)
    ax.set_ylim(0.45, 1.05)
    ax.set_ylabel('Confidence', color='white', fontsize=8)
    ax.tick_params(colors='white', labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor('#1a2332')

    # Stats
    pct_above_90 = (p_dom >= 0.90).mean()
    pct_above_70 = (p_dom >= 0.70).mean()
    pct_below_60 = (p_dom < 0.60).mean()
    ax.set_title(
        f'{horizon} | ≥90%: {pct_above_90:.1%}  ≥70%: {pct_above_70:.1%}  <60%: {pct_below_60:.1%}',
        color='white', fontsize=9
    )
    print(f'{horizon}: ≥90%={pct_above_90:.1%}  ≥70%={pct_above_70:.1%}  <60%={pct_below_60:.1%}  std={p_dom.std():.3f}')

plt.suptitle('Prediction Confidence Over Time — EURUSD 2022-2025\nGold line = 70% threshold',
             color='white', fontsize=11)
plt.tight_layout()
plt.show()

---
## TEST 3 — Pair-by-Pair Edge Concentration

**Hypothesis:** The edge may not be uniform. Some pairs may drive all the alpha while others are noise.

In [ ]:
# TEST 3 — Pair-by-pair accuracy breakdown

df_test = df[df.index >= '2022-01-01']

orig_models = {}
orig_calibrators = {}
for horizon in HORIZONS:
    orig_calibrators[horizon] = joblib.load(MODELS_DIR / f'calibrator_{horizon}.joblib')
    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        bundle = joblib.load(MODELS_DIR / f'model_{horizon}_Q{int(q*100)}.joblib')
        orig_models[(horizon, q_name)] = bundle['model']

print('PAIR-BY-PAIR ACCURACY — Test Set 2022-2025')
print('=' * 70)

results = []

for horizon, step in [('1H',1),('4H',4),('1D',24),('7D',168)]:
    print(f'\nHorizon: {horizon}')
    print(f'  {"Pair":<10} {"Overall acc":<14} {"≥70% acc":<14} {"≥70% freq":<12} {"EV"}')
    print(f'  {"-"*58}')

    for pair in PAIRS:
        df_pair = df_test[df_test['pair'] == pair].iloc[::step]
        if len(df_pair) < 50:
            continue

        p_cal, p_dom, correct, y = get_predictions(
            df_pair, horizon, orig_models, orig_calibrators[horizon]
        )

        overall_acc = correct.mean()
        acc_70, n_70 = accuracy_at_threshold(p_dom, correct, 0.70)
        freq_70 = (p_dom >= 0.70).mean()

        avg_move = np.abs(y[p_dom >= 0.70]).mean() * 100 if (p_dom >= 0.70).sum() > 10 else float('nan')
        ev = (acc_70 * avg_move) - ((1 - acc_70) * avg_move) - 0.028 if not np.isnan(acc_70) else float('nan')

        print(f'  {pair:<10} {overall_acc:<14.1%} {acc_70:<14.1%} {freq_70:<12.1%} {ev:+.4f}%')
        results.append({'horizon': horizon, 'pair': pair, 'overall_acc': overall_acc,
                        'acc_70': acc_70, 'freq_70': freq_70, 'ev': ev})

df_results = pd.DataFrame(results)
print(f'\nBest pair overall: {df_results.groupby("pair")["overall_acc"].mean().idxmax()}')
print(f'Best pair by EV:   {df_results.groupby("pair")["ev"].mean().idxmax()}')

---
## TEST 4 — Proper Momentum Baseline

**Hypothesis:** A proper linear regression momentum model might capture 10-15% of the edge, leaving true unique alpha at 10-15% rather than 25%.

In [ ]:
# TEST 4 — Linear regression momentum baseline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

df_train_lr = df[(df.index >= '2009-01-01') & (df.index < '2020-01-01')]
df_test_lr  = df[df.index >= '2022-01-01']
df_eu_train = df_train_lr[df_train_lr['pair'] == 'EURUSD']
df_eu_test  = df_test_lr[df_test_lr['pair'] == 'EURUSD']

# Momentum features: last N returns + volatility
momentum_cols = [c for c in feature_cols if 'log_ret' in c and '1H' in c]
momentum_cols += [c for c in feature_cols if 'rsi_14_1H' in c or 'macd_hist_1H' in c or 'rvol_24_1H' in c]
print(f'Momentum features: {len(momentum_cols)}')
print(momentum_cols)

print('\nLR MOMENTUM BASELINE vs LUMENY MODEL — EURUSD, 2022-2025')
print('=' * 65)

for horizon, step in [('1H',1),('4H',4),('1D',24),('7D',168)]:
    y_train = (df_eu_train[f'label_{horizon}'].values < 0).astype(int)
    y_test  = (df_eu_test.iloc[::step][f'label_{horizon}'].values < 0).astype(int)

    X_train_lr = df_eu_train[momentum_cols].ffill().fillna(0).values
    X_test_lr  = df_eu_test.iloc[::step][momentum_cols].ffill().fillna(0).values

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train_lr)
    X_test_s  = scaler.transform(X_test_lr)

    lr = LogisticRegression(max_iter=1000, C=0.1)
    lr.fit(X_train_s, y_train)
    lr_acc = (lr.predict(X_test_s) == y_test).mean()

    # LumenY model accuracy
    bundle = joblib.load(MODELS_DIR / f'model_{horizon}_Q50.joblib')
    lm_models = {}
    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        b = joblib.load(MODELS_DIR / f'model_{horizon}_Q{int(q*100)}.joblib')
        lm_models[(horizon, q_name)] = b['model']
    cal = joblib.load(MODELS_DIR / f'calibrator_{horizon}.joblib')

    p_cal, p_dom, correct, _ = get_predictions(
        df_eu_test.iloc[::step], horizon, lm_models, cal
    )
    lm_acc = correct.mean()
    alpha  = lm_acc - lr_acc

    print(f'{horizon}: LR baseline={lr_acc:.1%}  LumenY={lm_acc:.1%}  Alpha={alpha:+.1%}')